In [ ]:
%%writefile program1.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <stdlib.h> // For rand()

// Matrix размер (N x N)
// 矩阵大小（N x N）

#define N 1024


// Размер блока потоков (TILE_DIM x TILE_DIM)
// 线程块大小（TILE_DIM x TILE_DIM）

#define TILE_DIM 32


// --- Kernel 1: Наивное умножение матриц, использующее только глобальную память
// --- Kernel 1：只使用全局内存的朴素矩阵乘法实现

__global__ void matrixMulNaive(float *A, float *B, float *C) {

    // Вычисление позиции (row, col) в матрице C, которую нужно заполнить
    // 计算矩阵 C 中需要写入的 (row, col) 位置
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    float Cvalue = 0.0f;

    // Для элемента C(row, col) перемножаем соответствующую строку A и столбец B
    // 对于 C(row, col)，计算 A 的对应行 与 B 的对应列 的乘积求和

    if (row < N && col < N) {
        for (int k = 0; k < N; ++k) {

            // Многократное чтение из глобальной памяти (низкая эффективность)
            // 多次从全局内存读取（效率较低）

            Cvalue += A[row * N + k] * B[k * N + col];
        }

        // Запись результата в C
        // 将结果写入 C
        C[row * N + col] = Cvalue;
    }
}


int main() {

    // Расчёт необходимого объёма памяти
    // 计算所需内存大小
    size_t bytes = N * N * sizeof(float);


    // 1. Выделение памяти на Host (CPU)
    // 1. 在主机（CPU）上分配内存

    float *h_A = (float *)malloc(bytes);
    float *h_B = (float *)malloc(bytes);
    float *h_C = (float *)malloc(bytes);


    // 2. Инициализация данных в Host памяти
    // 2. 初始化主机内存中的数据

    for (int i = 0; i < N * N; ++i) {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
    }

    // 3. Выделение памяти на Device (GPU)
    // 3. 在设备（GPU）上分配内存

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);


    // 4. Копирование данных с Host на Device (H2D)
    // 4. 将数据从主机复制到设备（H2D）

    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);


    // 5. Настройка Grid и Block для запуска kernel
    // 5. 为 kernel 启动配置 Grid 和 Block 维度

    dim3 threadsPerBlock(TILE_DIM, TILE_DIM);
    dim3 numBlocks(N / threadsPerBlock.x, N / threadsPerBlock.y);


    // 6. Создание CUDA событий для измерения времени
    // 6. 创建 CUDA 事件用于计时

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // ---- Запуск Kernel 1 и измерение времени исполнения
    // ---- 启动 Kernel 1 并测量执行时间

    cudaEventRecord(start);

    matrixMulNaive<<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop); // Ждать завершения kernel
                                // 等待 kernel 执行完成

    float timeNaive = 0;
    cudaEventElapsedTime(&timeNaive, start, stop);
    printf("Global Memory (Naive) Kernel Time: %.3f ms\n", timeNaive);


    // 7. (Необязательно) Копирование результата обратно Device → Host
    // 7. （可选）将结果从设备复制回主机（D2H）

    // cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);
    // printf("Result C[0] = %f\n", h_C[0]);


    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}


Writing program1.cu


In [ ]:
!nvcc program1.cu -o program1

In [ ]:
!./program1

Global Memory (Naive) Kernel Time: 8.297 ms


### Результат эксперимента: умножение матриц (Naive Global Memory)

При выполнении наивного ядра умножения матриц размером 1024×1024, использующего только глобальную память без каких-либо оптимизаций, было получено время **8.297 ms**.  




In [ ]:
%%writefile program2.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <stdlib.h>

#define N 1024
#define TILE_DIM 32

// --- Kernel 2: Умножение матриц с использованием Shared Memory (Tiled)
// --- Kernel 2：使用 Shared Memory（分块法）的矩阵乘法

__global__ void matrixMulShared(float *A, float *B, float *C) {

    // 1. Выделение shared memory для одного блока
    // 1. 为每个线程块分配 shared memory
    __shared__ float As[TILE_DIM][TILE_DIM];
    __shared__ float Bs[TILE_DIM][TILE_DIM];

    // Индексы блока и потока
    // Block 与 Thread 的索引
    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    // Координаты (row, col) в матрице C, которые нужно вычислить
    // 需要计算的 C 矩阵 (row, col) 位置
    int row = by * TILE_DIM + ty;
    int col = bx * TILE_DIM + tx;

    float Cvalue = 0.0f;

    // 2. Цикл по всем тайлам (блокам данных)
    // 2. 循环加载所有 tile（分块）
    for (int m = 0; m < N / TILE_DIM; ++m) {

        // 3. Копирование данных из глобальной памяти в shared memory
        // 3. 将数据从 global memory 加载到 shared memory
        As[ty][tx] = (row < N && (m * TILE_DIM + tx) < N)
                     ? A[row * N + (m * TILE_DIM + tx)]
                     : 0;

        Bs[ty][tx] = ((m * TILE_DIM + ty) < N && col < N)
                     ? B[(m * TILE_DIM + ty) * N + col]
                     : 0;

        // 4. Синхронизация всех потоков в блоке перед вычислениями
        // 4. 所有线程同步，确保 tile 数据全部加载完毕
        __syncthreads();

        // 5. Вычисление частичного результата, используя данные из shared memory
        // 5. 使用 shared memory 中的数据执行部分矩阵乘法
        for (int k = 0; k < TILE_DIM; ++k) {
            Cvalue += As[ty][k] * Bs[k][tx];
        }

        // 6. Синхронизация перед переходом к следующему тайлу
        // 6. 在加载下一个 tile 前再次同步
        __syncthreads();
    }

    // 7. Запись окончательного результата в глобальную память C
    // 7. 将最终计算结果写回全局内存 C
    if (row < N && col < N) {
        C[row * N + col] = Cvalue;
    }
}


int main() {

    size_t bytes = N * N * sizeof(float);

    // Выделение памяти на Host
    // 主机内存分配
    float *h_A = (float *)malloc(bytes);
    float *h_B = (float *)malloc(bytes);
    float *h_C = (float *)malloc(bytes);

    // Инициализация данных
    // 初始化数据
    for (int i = 0; i < N * N; ++i) {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
    }

    // Выделение памяти на Device
    // 分配 GPU 设备内存
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    // Копирование данных Host → Device
    // 将数据从主机复制到设备（H2D）
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    // Настройка Grid и Block
    // 配置 Grid 和 Block 维度
    dim3 threadsPerBlock(TILE_DIM, TILE_DIM);
    dim3 numBlocks(N / threadsPerBlock.x, N / threadsPerBlock.y);

    // Создание CUDA событий для измерения времени
    // 创建 CUDA event 用于计时
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);


    // ---- Измерение времени выполнения Kernel 2 (Shared Memory)
    // ---- 计时 Kernel 2（Shared Memory 优化版）

    cudaEventRecord(start);

    matrixMulShared<<<numBlocks, threadsPerBlock>>>(d_A, d_B, d_C);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float timeShared = 0;
    cudaEventElapsedTime(&timeShared, start, stop);
    printf("Shared Memory (Tiled) Kernel Time: %.3f ms\n", timeShared);

    // Очистка памяти
    // 清理内存
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}


Writing program2.cu


In [ ]:
!nvcc program2.cu -o program2

In [ ]:
!./program2

Shared Memory (Tiled) Kernel Time: 7.296 ms


### Результат эксперимента: умножение матриц с использованием Shared Memory (Tiled)

При выполнении tiled-реализации умножения матриц размером 1024×1024, использующей Shared Memory для уменьшения числа обращений к глобальной памяти, было получено время **7.296 ms**.  




In [ ]:
%%writefile program3.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <math.h> // For sinf

// Размер массива

#define N (1024 * 1024)

// Размер блока потоков

#define BLOCK_SIZE 256


// --- Kernel 3: Чтение данных из глобальной памяти
// --- Kernel 3：从 Global Memory 读取数据

__global__ void readGlobal(float *d_input, float *d_output) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        // Чтение из глобальной памяти (d_input) и запись в d_output
        // 从全局内存 d_input 读取，并写入 d_output

        d_output[idx] = d_input[idx];
    }
}

int main() {

    size_t bytes = N * sizeof(float);


    // Host memory
    // 主机内存

    float *h_input = (float *)malloc(bytes);

    // Подготовка данных функции, например sin(x)
    // 准备函数数据，例如 sin(x)
    for (int i = 0; i < N; ++i) {
        h_input[i] = sinf((float)i * 0.01f);
    }

    // Device memory
    // 设备(GPU)内存

    float *d_input, *d_output;
    cudaMalloc(&d_input, bytes);
    cudaMalloc(&d_output, bytes);


    // Копирование данных Host → Device
    // 将数据从 Host 复制到 Device（H2D）

    cudaMemcpy(d_input, h_input, bytes, cudaMemcpyHostToDevice);


    // Grid / Block конфигурация
    // Grid / Block 配置

    dim3 threadsPerBlock(BLOCK_SIZE);
    dim3 numBlocks((N + threadsPerBlock.x - 1) / threadsPerBlock.x);


    // CUDA события для измерения времени
    // 创建 CUDA event 用于计时

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);


    // --- Измерение времени Kernel 3 (чтение из глобальной памяти)
    // --- 计时 Kernel 3（全局内存读取）

    cudaEventRecord(start);

    readGlobal<<<numBlocks, threadsPerBlock>>>(d_input, d_output);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float timeGlobalRead = 0;
    cudaEventElapsedTime(&timeGlobalRead, start, stop);
    printf("Global Memory Read Time: %.3f ms\n", timeGlobalRead);


    // Очистка памяти
    // 清理内存

    cudaFree(d_input);
    cudaFree(d_output);

    free(h_input);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}



Writing program3.cu


In [ ]:
!nvcc program3.cu -o program3

In [ ]:
!./program3

Global Memory Read Time: 7.454 ms


### Результат эксперимента: чтение из Global Memory

При выполнении ядра, последовательно считывающего одномерный массив размером 1M элементов из глобальной памяти, было зафиксировано время **7.454 ms**.  



In [ ]:
%%writefile program4_fixed.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <math.h>

#define N (1024 * 1024)
#define BLOCK_SIZE 256


// --- Kernel 4 (Modern API): Принятие Texture Object в качестве параметра
// --- Kernel 4 (现代 API)：以 Texture Object 作为参数读取纹理内存

__global__ void readTexture(cudaTextureObject_t texObj, float *d_output) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        // Чтение из texture object (tex1Dfetch<float>...)
        // 从 texture object 中读取数据（tex1Dfetch<float>...）
        d_output[idx] = tex1Dfetch<float>(texObj, idx);
    }
}

int main() {

    size_t bytes = N * sizeof(float);


    // Host memory
    // 主机内存

    float *h_input = (float *)malloc(bytes);

    // Подготовка функции, например sin(x)
    // 准备函数数据，例如 sin(x)
    for (int i = 0; i < N; ++i) {
        h_input[i] = sinf((float)i * 0.01f);
    }


    // Device memory
    // 设备（GPU）内存

    float *d_input, *d_output;
    cudaMalloc(&d_input, bytes);
    cudaMalloc(&d_output, bytes);

    // Копирование Host → Device
    // 将数据从主机复制到设备（H2D）
    cudaMemcpy(d_input, h_input, bytes, cudaMemcpyHostToDevice);


    // --- 1. Создание Texture Object (современный API)
    // --- 1. 创建 Texture Object（现代 API）



    // Настройка источника данных (resource)
    // 指定纹理对象的数据源（resource）

    cudaResourceDesc resDesc;
    memset(&resDesc, 0, sizeof(resDesc));
    resDesc.resType = cudaResourceTypeLinear;
    resDesc.res.linear.devPtr = d_input;
    resDesc.res.linear.sizeInBytes = bytes;
    resDesc.res.linear.desc.f = cudaChannelFormatKindFloat; // Тип данных — float
                                                             // 数据类型为 float
    resDesc.res.linear.desc.x = 32; // 32 бита для float
                                    // float 为 32 位


    // Настройка свойств Texture Object
    // 设置纹理对象的属性

    cudaTextureDesc texDesc;
    memset(&texDesc, 0, sizeof(texDesc));
    texDesc.readMode = cudaReadModeElementType;
    // Читать данные как float без преобразования
    // 以 float 原始格式读取，不做任何转换


    // Создание Texture Object
    // 创建纹理对象 texObj

    cudaTextureObject_t texObj = 0;
    cudaCreateTextureObject(&texObj, &resDesc, &texDesc, NULL);



    dim3 threadsPerBlock(BLOCK_SIZE);
    dim3 numBlocks((N + threadsPerBlock.x - 1) / threadsPerBlock.x);


    // CUDA события для замера времени
    // 创建 CUDA event 用于计时
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);


    // --- Измерение времени Kernel 4 (Texture Memory)
    // --- 计时 Kernel 4（纹理内存读取）

    cudaEventRecord(start);

    // Передача texObj в kernel является обязательной
    // 调用 kernel 时必须传入 texObj
    readTexture<<<numBlocks, threadsPerBlock>>>(texObj, d_output);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float timeTextureRead = 0;
    cudaEventElapsedTime(&timeTextureRead, start, stop);
    printf("Texture Memory (Modern API) Read Time: %.3f ms\n", timeTextureRead);


    // 4. Уничтожение Texture Object
    // 4. 销毁纹理对象
    cudaDestroyTextureObject(texObj);

    // Выделенную память освободить
    // 释放内存
    cudaFree(d_input);
    cudaFree(d_output);
    free(h_input);


    // 5. Корректное уничтожение CUDA events
    // 5. 正确销毁 CUDA event（逐个销毁）

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    return 0;
}


Writing program4_fixed.cu


In [ ]:
!nvcc program4_fixed.cu -o program4_fixed

In [ ]:
!./program4_fixed

Texture Memory (Modern API) Read Time: 7.142 ms


### Результат эксперимента: чтение через Texture Memory (Modern API)

При выполнении ядра, использующего Texture Object для чтения массива из 1M элементов, было получено время **7.142 ms**.  

